In [1]:
!pip install lifelines shap lime captum scikit-survival --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle

CHECKPOINT_DIR = '/content/drive/MyDrive/pancreatic_paper_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(name, obj):
    with open(f'{CHECKPOINT_DIR}/{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)
    print(f"Saved checkpoint: {name}")

def load_checkpoint(name):
    path = f'{CHECKPOINT_DIR}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            print(f"Loaded checkpoint: {name}")
            return pickle.load(f)
    return None

In [2]:
import time
import json
from itertools import combinations

import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler

from lifelines.utils import concordance_index
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

import shap
from lime.lime_tabular import LimeTabularExplainer
from captum.attr import IntegratedGradients
from scipy.stats import spearmanr

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
cv_state = load_checkpoint('cv_state')
if cv_state:
    results_cv = cv_state['results_cv']
    fold_fusion_models = cv_state['fold_fusion_models']
    fold_scalers = cv_state['fold_scalers']
    fold_test_indices = cv_state['fold_test_indices']

model_state = load_checkpoint('model_full_state')
if model_state:
    model_full = GatedFusionModel()
    model_full.load_state_dict(model_state['model_full_state_dict'])
    model_full.eval()
    sc_full, sg_full = model_state['sc_full'], model_state['sg_full']
    X_all = model_state['X_all']

shap_state = load_checkpoint('shap_state')
if shap_state:
    shap_values = shap_state['shap_values']
    top20_genes = shap_state['top20_genes']

In [3]:
df = pd.read_csv('tcga_paad_clean.csv')

stage_map = {
    'STAGE I': 1, 'STAGE IA': 1, 'STAGE IB': 1,
    'STAGE II': 2, 'STAGE IIA': 2, 'STAGE IIB': 2,
    'STAGE III': 3, 'STAGE IV': 4
}
df['STAGE_ENCODED'] = df['AJCC_PATHOLOGIC_TUMOR_STAGE'].map(stage_map).fillna(2)

gene_cols     = df.columns[5:1005].tolist()
clinical_cols = ['AGE', 'STAGE_ENCODED']

X_clinical = df[clinical_cols].values.astype(np.float32)
X_genomic  = df[gene_cols].values.astype(np.float32)
T          = df['OS_MONTHS'].values.astype(np.float32)
E          = df['OS_STATUS'].values.astype(np.float32)

# Clip genomic outliers before scaling
X_genomic = np.clip(X_genomic,
                    np.percentile(X_genomic, 1, axis=0),
                    np.percentile(X_genomic, 99, axis=0))

CLIN_DIM = X_clinical.shape[1]
feature_names = clinical_cols + gene_cols

print(f"Clinical features: {X_clinical.shape[1]}")
print(f"Genomic features:  {X_genomic.shape[1]}")
print(f"Patients:          {len(df)}")

Clinical features: 2
Genomic features:  1000
Patients:          177


In [4]:
def cox_ph_loss(risk, T, E):
    order      = torch.argsort(T, descending=True)
    risk       = risk[order]
    E          = E[order]
    log_cumsum = torch.logcumsumexp(risk, dim=0)
    return -torch.mean((risk - log_cumsum) * E)

In [5]:
class ClinicalMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)


class GenomicMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1000, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)


class GatedFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.clin_encoder = nn.Sequential(
            nn.Linear(2, 32), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU()
        )
        self.gen_encoder = nn.Sequential(
            nn.Linear(1000, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4)
        )
        self.gate_clin = nn.Sequential(nn.Linear(192, 64), nn.Sigmoid())
        self.gate_gen  = nn.Sequential(nn.Linear(192, 128), nn.Sigmoid())
        self.head      = nn.Linear(192, 1)

    def forward(self, x_clin, x_gen):
        h_clin  = self.clin_encoder(x_clin)
        h_gen   = self.gen_encoder(x_gen)
        h       = torch.cat([h_clin, h_gen], dim=1)
        g_clin  = self.gate_clin(h)
        g_gen   = self.gate_gen(h)
        h_fused = torch.cat([h_clin * g_clin, h_gen * g_gen], dim=1)
        return self.head(h_fused).squeeze(1)

    def get_gates(self, x_clin, x_gen):
        h_clin = self.clin_encoder(x_clin)
        h_gen  = self.gen_encoder(x_gen)
        h      = torch.cat([h_clin, h_gen], dim=1)
        return self.gate_clin(h), self.gate_gen(h)

In [9]:
def train_model(model, Xc, Xg, T, E, epochs=300, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    Xc = torch.tensor(Xc); Xg = torch.tensor(Xg)
    Tt = torch.tensor(T);  Et = torch.tensor(E)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        if isinstance(model, ClinicalMLP):
            risk = model(Xc)
        elif isinstance(model, GenomicMLP):
            risk = model(Xg)
        else:
            # any fusion model (GatedFusionModel, ConcatFusionModel, etc.) takes both inputs
            risk = model(Xc, Xg)
        loss = cox_ph_loss(risk, Tt, Et)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    return model

In [10]:
skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_cv = {'clinical': [], 'genomic': [], 'fusion': []}

fold_fusion_models = []
fold_scalers       = []
fold_test_indices  = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_clinical, E)):
    sc = RobustScaler(); sg = RobustScaler()
    Xc_tr = sc.fit_transform(X_clinical[train_idx]); Xc_te = sc.transform(X_clinical[test_idx])
    Xg_tr = sg.fit_transform(X_genomic[train_idx]);  Xg_te = sg.transform(X_genomic[test_idx])
    T_tr, E_tr = T[train_idx], E[train_idx]
    T_te, E_te = T[test_idx],  E[test_idx]

    for name, model_cls in [('clinical', ClinicalMLP), ('genomic', GenomicMLP), ('fusion', GatedFusionModel)]:
        torch.manual_seed(42)
        model = model_cls()
        model = train_model(model, Xc_tr, Xg_tr, T_tr, E_tr)
        model.eval()
        with torch.no_grad():
            if name == 'clinical':
                risk = model(torch.tensor(Xc_te)).numpy()
            elif name == 'genomic':
                risk = model(torch.tensor(Xg_te)).numpy()
            else:
                risk = model(torch.tensor(Xc_te), torch.tensor(Xg_te)).numpy()
        ci = concordance_index(T_te, -risk, E_te)
        results_cv[name].append(ci)
        print(f"Fold {fold+1} | {name:10s} C-index: {ci:.4f}")

        if name == 'fusion':
            fold_fusion_models.append(model)
            fold_scalers.append((sc, sg))
            fold_test_indices.append(test_idx)

print("\n--- Final CV Results (report these in your paper's Table 1) ---")
for name, scores in results_cv.items():
    print(f"{name:10s}: {np.mean(scores):.4f} +/- {np.std(scores):.4f}")

save_checkpoint('cv_state', {
    'results_cv': results_cv,
    'fold_fusion_models': fold_fusion_models,
    'fold_scalers': fold_scalers,
    'fold_test_indices': fold_test_indices,
})

Fold 1 | clinical   C-index: 0.5375
Fold 1 | genomic    C-index: 0.5736
Fold 1 | fusion     C-index: 0.5633
Fold 2 | clinical   C-index: 0.4893
Fold 2 | genomic    C-index: 0.4973
Fold 2 | fusion     C-index: 0.5214
Fold 3 | clinical   C-index: 0.6087
Fold 3 | genomic    C-index: 0.5924
Fold 3 | fusion     C-index: 0.6440
Fold 4 | clinical   C-index: 0.6400
Fold 4 | genomic    C-index: 0.6000
Fold 4 | fusion     C-index: 0.5400
Fold 5 | clinical   C-index: 0.7012
Fold 5 | genomic    C-index: 0.7027
Fold 5 | fusion     C-index: 0.6817

--- Final CV Results (report these in your paper's Table 1) ---
clinical  : 0.5953 +/- 0.0748
genomic   : 0.5932 +/- 0.0657
fusion    : 0.5901 +/- 0.0620


In [11]:
class ConcatFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.clin_encoder = nn.Sequential(
            nn.Linear(2, 32), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU()
        )
        self.gen_encoder = nn.Sequential(
            nn.Linear(1000, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4)
        )
        self.head = nn.Linear(192, 1)

    def forward(self, x_clin, x_gen):
        h_clin = self.clin_encoder(x_clin)
        h_gen  = self.gen_encoder(x_gen)
        h      = torch.cat([h_clin, h_gen], dim=1)
        return self.head(h).squeeze(1)


concat_scores = []
for fold, (train_idx, test_idx) in enumerate(skf.split(X_clinical, E)):
    sc = RobustScaler(); sg = RobustScaler()
    Xc_tr = sc.fit_transform(X_clinical[train_idx]); Xc_te = sc.transform(X_clinical[test_idx])
    Xg_tr = sg.fit_transform(X_genomic[train_idx]);  Xg_te = sg.transform(X_genomic[test_idx])
    T_tr, E_tr = T[train_idx], E[train_idx]
    T_te, E_te = T[test_idx],  E[test_idx]

    torch.manual_seed(42)
    m = ConcatFusionModel()
    m = train_model(m, Xc_tr, Xg_tr, T_tr, E_tr)
    m.eval()
    with torch.no_grad():
        risk = m(torch.tensor(Xc_te), torch.tensor(Xg_te)).numpy()
    ci = concordance_index(T_te, -risk, E_te)
    concat_scores.append(ci)
    print(f"Fold {fold+1} | concat_fusion C-index: {ci:.4f}")

print(f"\nconcat_fusion: {np.mean(concat_scores):.4f} +/- {np.std(concat_scores):.4f}")

Fold 1 | concat_fusion C-index: 0.5711
Fold 2 | concat_fusion C-index: 0.5267
Fold 3 | concat_fusion C-index: 0.6304
Fold 4 | concat_fusion C-index: 0.5486
Fold 5 | concat_fusion C-index: 0.6877

concat_fusion: 0.5929 +/- 0.0587


In [12]:
sc_full = RobustScaler(); sg_full = RobustScaler()
Xc_full = sc_full.fit_transform(X_clinical).astype(np.float32)
Xg_full = sg_full.fit_transform(X_genomic).astype(np.float32)

torch.manual_seed(42)
model_full = GatedFusionModel()
model_full = train_model(model_full, Xc_full, Xg_full, T, E, epochs=300)
model_full.eval()

X_all = np.hstack([Xc_full, Xg_full]).astype(np.float32)

def model_predict(X):
    X = X.astype(np.float32)
    Xc = torch.tensor(X[:, :CLIN_DIM])
    Xg = torch.tensor(X[:, CLIN_DIM:])
    with torch.no_grad():
        return model_full(Xc, Xg).numpy()

print("Final model trained. X_all shape:", X_all.shape)

save_checkpoint('model_full_state', {
    'model_full_state_dict': model_full.state_dict(),
    'sc_full': sc_full,
    'sg_full': sg_full,
    'X_all': X_all,
})

Final model trained. X_all shape: (177, 1002)


In [13]:
t0 = time.time()
background   = shap.sample(X_all, 30)
explainer    = shap.KernelExplainer(model_predict, background)
shap_values  = explainer.shap_values(X_all, nsamples=200, l1_reg='num_features(50)')
shap_runtime = time.time() - t0
print(f"SHAP runtime: {shap_runtime:.1f}s")

shap.summary_plot(shap_values, X_all, feature_names=feature_names, max_display=20, show=False)
plt.savefig('shap_beeswarm.png', dpi=500, bbox_inches='tight')
plt.close()
print("Saved: shap_beeswarm.png")

shap_clin = np.abs(shap_values[:, :CLIN_DIM]).sum(axis=1)
shap_gen  = np.abs(shap_values[:, CLIN_DIM:]).sum(axis=1)
total     = shap_clin + shap_gen
clinical_pct = (shap_clin / total).mean() * 100
genomic_pct  = (shap_gen  / total).mean() * 100
print(f"\nClinical attribution: {clinical_pct:.1f}%")
print(f"Genomic attribution:  {genomic_pct:.1f}%")

plt.figure(figsize=(5, 5))
plt.bar(['Clinical', 'Genomic'], [clinical_pct, genomic_pct], color=['#4C72B0', '#DD8452'])
plt.ylabel('Mean SHAP Attribution (%)')
plt.title('Modality-Level SHAP Attribution\n(TCGA-PAAD)')
for i, v in enumerate([clinical_pct, genomic_pct]):
    plt.text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold')
plt.savefig('modality_attribution.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: modality_attribution.png")

gene_shap   = np.abs(shap_values[:, CLIN_DIM:]).mean(axis=0)
gene_shap_c = np.clip(gene_shap, 0, np.percentile(gene_shap, 99))
top20_idx   = np.argsort(gene_shap_c)[::-1][:20]
top20_genes = [(gene_cols[i], gene_shap_c[i]) for i in top20_idx]
print("\nTop 20 SHAP genes:")
for gene, val in top20_genes:
    print(f"  {gene}: {val:.4f}")

  0%|          | 0/177 [00:00<?, ?it/s]

SHAP runtime: 41.9s


/tmp/ipykernel_11381/2704649612.py:8: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_all, feature_names=feature_names, max_display=20, show=False)


Saved: shap_beeswarm.png

Clinical attribution: 1.8%
Genomic attribution:  98.2%
Saved: modality_attribution.png

Top 20 SHAP genes:
  CCND1: 0.8851
  CTSK: 0.8851
  DEFB1: 0.8851
  PTMS: 0.8851
  BCAS1: 0.8851
  KRT7: 0.8851
  CLCA1: 0.8851
  PDZK1IP1: 0.8851
  MUC2: 0.8851
  LOXL2: 0.8851
  ITLN1: 0.8850
  REG3G: 0.7411
  SPINK4: 0.7032
  IGFBP3: 0.6996
  CAPG: 0.6629
  CNN2: 0.6622
  MIAT: 0.5945
  CFH: 0.5881
  CTSD: 0.5787
  ITPR3: 0.4698


In [14]:
cosmic_paad = {
    'KRAS', 'TP53', 'SMAD4', 'CDKN2A', 'ARID1A',
    'ROBO2', 'KDM6A', 'RNF43', 'TGFBR2', 'BRAF',
    'GNAS', 'MYC', 'PTEN', 'RB1', 'STK11'
}

print("Top 20 SHAP genes -- COSMIC PDAC driver annotation:")
print(f"{'Rank':<6}{'Gene':<15}{'Mean |SHAP|':<15}{'COSMIC Driver'}")
print("-" * 55)
for rank, (gene, val) in enumerate(top20_genes, 1):
    is_driver = "YES" if gene in cosmic_paad else "-"
    print(f"{rank:<6}{gene:<15}{val:<15.4f}{is_driver}")

overlap = [g for g, _ in top20_genes if g in cosmic_paad]
print(f"\nOverlap with COSMIC PDAC drivers: {len(overlap)}/20 -> {overlap}")

Top 20 SHAP genes -- COSMIC PDAC driver annotation:
Rank  Gene           Mean |SHAP|    COSMIC Driver
-------------------------------------------------------
1     CCND1          0.8851         -
2     CTSK           0.8851         -
3     DEFB1          0.8851         -
4     PTMS           0.8851         -
5     BCAS1          0.8851         -
6     KRT7           0.8851         -
7     CLCA1          0.8851         -
8     PDZK1IP1       0.8851         -
9     MUC2           0.8851         -
10    LOXL2          0.8851         -
11    ITLN1          0.8850         -
12    REG3G          0.7411         -
13    SPINK4         0.7032         -
14    IGFBP3         0.6996         -
15    CAPG           0.6629         -
16    CNN2           0.6622         -
17    MIAT           0.5945         -
18    CFH            0.5881         -
19    CTSD           0.5787         -
20    ITPR3          0.4698         -

Overlap with COSMIC PDAC drivers: 0/20 -> []


In [15]:
with torch.no_grad():
    all_risk = model_full(torch.tensor(Xc_full), torch.tensor(Xg_full)).numpy()

median_risk = np.median(all_risk)
high_risk   = all_risk >= median_risk
low_risk    = all_risk <  median_risk

results_lr = logrank_test(T[high_risk], T[low_risk], E[high_risk], E[low_risk])
print(f"Log-rank p-value: {results_lr.p_value:.6f}")

kmf_high = KaplanMeierFitter()
kmf_low  = KaplanMeierFitter()

plt.figure(figsize=(8, 6))
ax = plt.gca()
kmf_high.fit(T[high_risk], E[high_risk], label='High Risk')
kmf_high.plot_survival_function(ax=ax, color='red')
kmf_low.fit(T[low_risk], E[low_risk], label='Low Risk')
kmf_low.plot_survival_function(ax=ax, color='blue')
plt.title('Kaplan-Meier Survival Curves\nHigh vs Low Risk -- TCGA-PAAD')
plt.xlabel('Overall Survival (Months)')
plt.ylabel('Survival Probability')
plt.savefig('km_curve.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: km_curve.png")

save_checkpoint('shap_state', {'shap_values': shap_values, 'top20_genes': top20_genes})

Log-rank p-value: 0.000000
Saved: km_curve.png


In [16]:
def get_gate_genomic_fraction(model, X_all_np):
    xc = torch.tensor(X_all_np[:, :CLIN_DIM].astype(np.float32))
    xg = torch.tensor(X_all_np[:, CLIN_DIM:].astype(np.float32))
    with torch.no_grad():
        g_clin, g_gen = model.get_gates(xc, xg)
    mg_clin = g_clin.mean().item()
    mg_gen  = g_gen.mean().item()
    return mg_gen / (mg_gen + mg_clin + 1e-10)


def make_numpy_predict_fn(model):
    def predict_fn(X):
        X = X.astype(np.float32)
        xc = torch.tensor(X[:, :CLIN_DIM])
        xg = torch.tensor(X[:, CLIN_DIM:])
        with torch.no_grad():
            return model(xc, xg).numpy()
    return predict_fn


class WrappedTorchModel(nn.Module):
    def __init__(self, base_model, clin_dim=CLIN_DIM):
        super().__init__()
        self.base = base_model
        self.clin_dim = clin_dim
    def forward(self, x):
        return self.base(x[:, :self.clin_dim], x[:, self.clin_dim:])


def compute_lime_global(model, X_all_np, feature_names, num_samples=300):
    t0 = time.time()
    predict_fn = make_numpy_predict_fn(model)
    explainer = LimeTabularExplainer(
        training_data=X_all_np, feature_names=feature_names,
        mode="regression", discretize_continuous=False,
    )
    n_features = X_all_np.shape[1]
    attributions = np.zeros_like(X_all_np, dtype=float)
    for i in range(X_all_np.shape[0]):
        exp = explainer.explain_instance(X_all_np[i], predict_fn, num_features=n_features, num_samples=num_samples)
        for feat_idx, weight in dict(exp.as_map()[1]).items():
            attributions[i, feat_idx] = weight
        if i % 30 == 0:
            print(f"    LIME: {i}/{X_all_np.shape[0]}")
    return np.mean(np.abs(attributions), axis=0), time.time() - t0


def compute_ig_global(model, X_all_np):
    t0 = time.time()
    wrapped = WrappedTorchModel(model)
    wrapped.eval()
    X_t = torch.tensor(X_all_np.astype(np.float32), requires_grad=True)
    baseline_vec = np.median(X_all_np, axis=0)
    baseline = torch.tensor(np.tile(baseline_vec, (X_all_np.shape[0], 1)).astype(np.float32))
    ig = IntegratedGradients(wrapped)
    attributions, _ = ig.attribute(X_t, baselines=baseline, n_steps=50, return_convergence_delta=True)
    return np.mean(np.abs(attributions.detach().numpy()), axis=0), time.time() - t0


def compute_pfi_global(model, X_all_np, T, E, n_repeats=10, random_state=42):
    t0 = time.time()
    rng = np.random.RandomState(random_state)
    predict_fn = make_numpy_predict_fn(model)
    baseline_preds = predict_fn(X_all_np)
    baseline_cindex = concordance_index(T, -baseline_preds, E)
    n_features = X_all_np.shape[1]
    importances = np.zeros(n_features)
    for f in range(n_features):
        drops = []
        for _ in range(n_repeats):
            X_perm = X_all_np.copy()
            X_perm[:, f] = rng.permutation(X_perm[:, f])
            perm_cindex = concordance_index(T, -predict_fn(X_perm), E)
            drops.append(baseline_cindex - perm_cindex)
        importances[f] = np.mean(drops)
        if f % 100 == 0:
            print(f"    PFI: {f}/{n_features}")
    return importances, time.time() - t0


def compute_pdp_global(model, X_all_np, candidate_idx, n_grid_points=10):
    t0 = time.time()
    predict_fn = make_numpy_predict_fn(model)
    full_vector = np.zeros(X_all_np.shape[1])
    for feat_idx in candidate_idx:
        grid = np.percentile(X_all_np[:, feat_idx], np.linspace(10, 90, n_grid_points))
        curve = []
        for val in grid:
            X_mod = X_all_np.copy()
            X_mod[:, feat_idx] = val
            curve.append(predict_fn(X_mod).mean())
        full_vector[feat_idx] = max(curve) - min(curve)
    return full_vector, time.time() - t0


def faithfulness_score(model, X_all_np, global_importance, T, E, top_k):
    predict_fn = make_numpy_predict_fn(model)
    baseline_values = np.median(X_all_np, axis=0)
    original_cindex = concordance_index(T, -predict_fn(X_all_np), E)
    top_k_idx = np.argsort(-global_importance)[:top_k]
    X_perturbed = X_all_np.copy()
    X_perturbed[:, top_k_idx] = baseline_values[top_k_idx]
    perturbed_cindex = concordance_index(T, -predict_fn(X_perturbed), E)
    return original_cindex - perturbed_cindex


def stability_score(per_fold_importances, top_k):
    top_sets = [set(np.argsort(-imp)[:top_k]) for imp in per_fold_importances]
    jaccards = []
    for a, b in combinations(top_sets, 2):
        inter, union = len(a & b), len(a | b)
        jaccards.append(inter / union if union > 0 else 0.0)
    return np.mean(jaccards), np.std(jaccards)


def correlation_aware_stability(X_genomic_full, per_fold_importances, top_k, clin_dim=CLIN_DIM, corr_threshold=0.8):
    n_genes = X_genomic_full.shape[1]
    corr_matrix = np.corrcoef(X_genomic_full.T)
    parent = list(range(n_genes))
    def find(x):
        while parent[x] != x: x = parent[x]
        return x
    def union(x, y):
        px, py = find(x), find(y)
        if px != py: parent[px] = py
    for i in range(n_genes):
        for j in range(i + 1, n_genes):
            if abs(corr_matrix[i, j]) > corr_threshold:
                union(i, j)
    feat_to_cluster = {clin_dim + i: find(i) for i in range(n_genes)}
    for i in range(clin_dim):
        feat_to_cluster[i] = f"clinical_{i}"
    cluster_top_sets = []
    for imp in per_fold_importances:
        top_k_idx = np.argsort(-imp)[:top_k]
        cluster_top_sets.append(set(feat_to_cluster[idx] for idx in top_k_idx))
    jaccards = []
    for a, b in combinations(cluster_top_sets, 2):
        inter, union_ = len(a & b), len(a | b)
        jaccards.append(inter / union_ if union_ > 0 else 0.0)
    return np.mean(jaccards), np.std(jaccards), len(set(feat_to_cluster.values()))


def gate_agreement_score(model, X_all_np, global_importance, clin_dim=CLIN_DIM):
    gate_genomic_fraction = get_gate_genomic_fraction(model, X_all_np)
    method_genomic_fraction = global_importance[clin_dim:].sum() / (global_importance.sum() + 1e-10)
    agreement = 1 - abs(gate_genomic_fraction - method_genomic_fraction)
    return agreement, gate_genomic_fraction, method_genomic_fraction

print("Comparison functions defined.")

Comparison functions defined.


In [17]:
# --- Compute global importances for LIME, IG, PFI, PDP (SHAP already have) ---
shap_global = np.mean(np.abs(shap_values), axis=0)

print("Computing LIME...")
lime_global, lime_runtime = compute_lime_global(model_full, X_all, feature_names)

print("Computing Integrated Gradients...")
ig_global, ig_runtime = compute_ig_global(model_full, X_all)

print("Computing PFI...")
pfi_global, pfi_runtime = compute_pfi_global(model_full, X_all, T, E)

print("Computing PDP (top-25 candidates only)...")
candidates = set()
for imp in [shap_global, lime_global, ig_global, pfi_global]:
    candidates |= set(np.argsort(-imp)[:25])
pdp_global, pdp_runtime = compute_pdp_global(model_full, X_all, list(candidates))

runtimes = {
    'SHAP': shap_runtime, 'LIME': lime_runtime, 'IG': ig_runtime,
    'PFI': pfi_runtime, 'PDP': pdp_runtime,
}
global_importances = {
    'SHAP': shap_global, 'LIME': lime_global, 'IG': ig_global,
    'PFI': pfi_global, 'PDP': pdp_global,
}
print("\nRuntimes (s):", runtimes)

Computing LIME...
    LIME: 0/177
    LIME: 30/177
    LIME: 60/177
    LIME: 90/177
    LIME: 120/177
    LIME: 150/177
Computing Integrated Gradients...
Computing PFI...
    PFI: 0/1002
    PFI: 100/1002
    PFI: 200/1002
    PFI: 300/1002
    PFI: 400/1002
    PFI: 500/1002
    PFI: 600/1002
    PFI: 700/1002
    PFI: 800/1002
    PFI: 900/1002
    PFI: 1000/1002
Computing PDP (top-25 candidates only)...

Runtimes (s): {'SHAP': 41.881622314453125, 'LIME': 29.300960779190063, 'IG': 0.5805099010467529, 'PFI': 58.0346040725708, 'PDP': 1.701991319656372}


In [18]:
# --- Faithfulness ---
comparison_results = {}
for name, imp in global_importances.items():
    for k in [5, 10]:
        comparison_results[f"{name}_faithfulness_top{k}"] = faithfulness_score(
            model_full, X_all, imp, T, E, top_k=k
        )

# --- Gate agreement ---
model_gate_genomic_fraction = get_gate_genomic_fraction(model_full, X_all)
for name, imp in global_importances.items():
    agreement, gate_frac, method_frac = gate_agreement_score(model_full, X_all, imp)
    comparison_results[f"{name}_gate_agreement"] = agreement
    comparison_results[f"{name}_genomic_fraction"] = method_frac
comparison_results["model_gate_genomic_fraction"] = model_gate_genomic_fraction

print("Faithfulness + gate agreement done.")
for k, v in comparison_results.items():
    print(f"  {k}: {v}")

Faithfulness + gate agreement done.
  SHAP_faithfulness_top5: 0.03156866999349106
  SHAP_faithfulness_top10: 0.03156866999349106
  LIME_faithfulness_top5: 0.017899761336515607
  LIME_faithfulness_top10: 0.05250596658711226
  IG_faithfulness_top5: 0.06118463875027125
  IG_faithfulness_top10: 0.06530700802777178
  PFI_faithfulness_top5: 0.03601648947711
  PFI_faithfulness_top10: 0.05565198524625736
  PDP_faithfulness_top5: 0.045454545454545525
  PDP_faithfulness_top10: 0.05185506617487534
  SHAP_gate_agreement: 0.6073138747329776
  SHAP_genomic_fraction: 0.9459847782770674
  LIME_gate_agreement: 0.5567778400202629
  LIME_genomic_fraction: 0.9965208129897821
  IG_gate_agreement: 0.571242094039917
  IG_genomic_fraction: 0.9820565581321716
  PFI_gate_agreement: 0.59212910078623
  PFI_genomic_fraction: 0.961169552223815
  PDP_gate_agreement: 0.7270364163031974
  PDP_genomic_fraction: 0.8262622367068476
  model_gate_genomic_fraction: 0.553298653010045


In [19]:
# --- Stability across folds (SLOW) ---
per_fold_lime, per_fold_ig, per_fold_pfi = [], [], []

for i, fold_model in enumerate(fold_fusion_models):
    test_idx = fold_test_indices[i]
    sc, sg = fold_scalers[i]
    Xc_te = sc.transform(X_clinical[test_idx])
    Xg_te = sg.transform(X_genomic[test_idx])
    X_fold = np.hstack([Xc_te, Xg_te]).astype(np.float32)
    T_fold, E_fold = T[test_idx], E[test_idx]

    print(f"Fold {i+1}/5 -- LIME...")
    lime_f, _ = compute_lime_global(fold_model, X_fold, feature_names, num_samples=150)
    per_fold_lime.append(lime_f)

    print(f"Fold {i+1}/5 -- IG...")
    ig_f, _ = compute_ig_global(fold_model, X_fold)
    per_fold_ig.append(ig_f)

    print(f"Fold {i+1}/5 -- PFI...")
    pfi_f, _ = compute_pfi_global(fold_model, X_fold, T_fold, E_fold, n_repeats=5)
    per_fold_pfi.append(pfi_f)

    print(f"Fold {i+1}/5 done.")

for name, per_fold in [('LIME', per_fold_lime), ('IG', per_fold_ig), ('PFI', per_fold_pfi)]:
    mean_j, std_j = stability_score(per_fold, top_k=10)
    comparison_results[f"{name}_stability_jaccard_mean"] = mean_j
    comparison_results[f"{name}_stability_jaccard_std"] = std_j
    mean_jc, std_jc, n_clust = correlation_aware_stability(X_genomic, per_fold, top_k=10)
    comparison_results[f"{name}_stability_corr_aware_mean"] = mean_jc
    comparison_results[f"{name}_stability_corr_aware_std"] = std_jc

print("\nStability scoring done.")

Fold 1/5 -- LIME...
    LIME: 0/36
    LIME: 30/36
Fold 1/5 -- IG...
Fold 1/5 -- PFI...
    PFI: 0/1002
    PFI: 100/1002
    PFI: 200/1002
    PFI: 300/1002
    PFI: 400/1002
    PFI: 500/1002
    PFI: 600/1002
    PFI: 700/1002
    PFI: 800/1002
    PFI: 900/1002
    PFI: 1000/1002
Fold 1/5 done.
Fold 2/5 -- LIME...
    LIME: 0/36
    LIME: 30/36
Fold 2/5 -- IG...
Fold 2/5 -- PFI...
    PFI: 0/1002
    PFI: 100/1002
    PFI: 200/1002
    PFI: 300/1002
    PFI: 400/1002
    PFI: 500/1002
    PFI: 600/1002
    PFI: 700/1002
    PFI: 800/1002
    PFI: 900/1002
    PFI: 1000/1002
Fold 2/5 done.
Fold 3/5 -- LIME...
    LIME: 0/35
    LIME: 30/35
Fold 3/5 -- IG...
Fold 3/5 -- PFI...
    PFI: 0/1002
    PFI: 100/1002
    PFI: 200/1002
    PFI: 300/1002
    PFI: 400/1002
    PFI: 500/1002
    PFI: 600/1002
    PFI: 700/1002
    PFI: 800/1002
    PFI: 900/1002
    PFI: 1000/1002
Fold 3/5 done.
Fold 4/5 -- LIME...
    LIME: 0/35
    LIME: 30/35
Fold 4/5 -- IG...
Fold 4/5 -- PFI...
    PFI: 0/1

In [24]:
# --- Final summary table + save to file for download ---
comparison_results['runtimes'] = runtimes

print(json.dumps(comparison_results, indent=2, default=str))

with open('xai_comparison_results.json', 'w') as f:
    json.dump(comparison_results, f, indent=2, default=str)

print("\nSaved: xai_comparison_results.json.")

# quick readable summary table
print("\n=== SUMMARY: FAITHFULNESS (top-10 removal, higher = more faithful) ===")
for name in ['SHAP', 'LIME', 'IG', 'PFI', 'PDP']:
    print(f"  {name:6s}: {comparison_results.get(name + '_faithfulness_top10', 'N/A')}")

print("\n=== SUMMARY: GATE AGREEMENT (closer to 1 = more aligned with model's own gates) ===")
for name in ['SHAP', 'LIME', 'IG', 'PFI', 'PDP']:
    print(f"  {name:6s}: {comparison_results.get(name + '_gate_agreement', 'N/A')}")

print("\n=== SUMMARY: STABILITY (raw Jaccard / correlation-aware Jaccard, higher = more stable) ===")
for name in ['LIME', 'IG', 'PFI']:
    raw = comparison_results.get(name + '_stability_jaccard_mean', 'N/A')
    corr_aware = comparison_results.get(name + '_stability_corr_aware_mean', 'N/A')
    print(f"  {name:6s}: raw={raw}  corr_aware={corr_aware}")
print("  SHAP  : (add if you have your original per-fold SHAP saved)")

print("\n=== RUNTIMES (seconds) ===")
for name, rt in runtimes.items():
    print(f"  {name:6s}: {rt}")

{
  "SHAP_faithfulness_top5": 0.03156866999349106,
  "SHAP_faithfulness_top10": 0.03156866999349106,
  "LIME_faithfulness_top5": 0.017899761336515607,
  "LIME_faithfulness_top10": 0.05250596658711226,
  "IG_faithfulness_top5": 0.06118463875027125,
  "IG_faithfulness_top10": 0.06530700802777178,
  "PFI_faithfulness_top5": 0.03601648947711,
  "PFI_faithfulness_top10": 0.05565198524625736,
  "PDP_faithfulness_top5": 0.045454545454545525,
  "PDP_faithfulness_top10": 0.05185506617487534,
  "SHAP_gate_agreement": 0.6073138747329776,
  "SHAP_genomic_fraction": 0.9459847782770674,
  "LIME_gate_agreement": 0.5567778400202629,
  "LIME_genomic_fraction": 0.9965208129897821,
  "IG_gate_agreement": "0.5712421",
  "IG_genomic_fraction": "0.98205656",
  "PFI_gate_agreement": 0.59212910078623,
  "PFI_genomic_fraction": 0.961169552223815,
  "PDP_gate_agreement": 0.7270364163031974,
  "PDP_genomic_fraction": 0.8262622367068476,
  "model_gate_genomic_fraction": 0.553298653010045,
  "LIME_stability_jaccar

In [25]:
# --- Per-fold SHAP (fills the missing stability comparison) ---
def compute_shap_global_per_fold(model, X_fold, nsamples=200):
    t0 = time.time()
    predict_fn = make_numpy_predict_fn(model)
    background = shap.sample(X_fold, min(30, X_fold.shape[0]))
    explainer = shap.KernelExplainer(predict_fn, background)
    shap_vals = explainer.shap_values(X_fold, nsamples=nsamples, l1_reg='num_features(50)')
    runtime = time.time() - t0
    return np.mean(np.abs(shap_vals), axis=0), runtime

per_fold_shap = []
shap_fold_runtimes = []

for i, fold_model in enumerate(fold_fusion_models):
    test_idx = fold_test_indices[i]
    sc, sg = fold_scalers[i]
    Xc_te = sc.transform(X_clinical[test_idx])
    Xg_te = sg.transform(X_genomic[test_idx])
    X_fold = np.hstack([Xc_te, Xg_te]).astype(np.float32)

    print(f"Fold {i+1}/5 -- SHAP...")
    shap_f, rt = compute_shap_global_per_fold(fold_model, X_fold)
    per_fold_shap.append(shap_f)
    shap_fold_runtimes.append(rt)
    print(f"Fold {i+1}/5 done ({rt:.1f}s).")

# --- Stability scoring, same functions as the other methods ---
mean_j, std_j = stability_score(per_fold_shap, top_k=10)
comparison_results['SHAP_stability_jaccard_mean'] = mean_j
comparison_results['SHAP_stability_jaccard_std'] = std_j

mean_jc, std_jc, n_clust = correlation_aware_stability(X_genomic, per_fold_shap, top_k=10)
comparison_results['SHAP_stability_corr_aware_mean'] = mean_jc
comparison_results['SHAP_stability_corr_aware_std'] = std_jc

comparison_results['SHAP_per_fold_runtimes'] = shap_fold_runtimes

print(f"\nSHAP stability -- raw Jaccard: {mean_j:.4f} +/- {std_j:.4f}")
print(f"SHAP stability -- corr-aware: {mean_jc:.4f} +/- {std_jc:.4f}")

# --- Re-save the complete results file ---
with open('xai_comparison_results_complete.json', 'w') as f:
    json.dump(comparison_results, f, indent=2, default=str)

print("\nSaved: xai_comparison_results_complete.json -- download and send this back.")

Fold 1/5 -- SHAP...


  0%|          | 0/36 [00:00<?, ?it/s]

Fold 1/5 done (7.7s).
Fold 2/5 -- SHAP...


  0%|          | 0/36 [00:00<?, ?it/s]

Fold 2/5 done (8.8s).
Fold 3/5 -- SHAP...


  0%|          | 0/35 [00:00<?, ?it/s]

Fold 3/5 done (7.9s).
Fold 4/5 -- SHAP...


  0%|          | 0/35 [00:00<?, ?it/s]

Fold 4/5 done (8.1s).
Fold 5/5 -- SHAP...


  0%|          | 0/35 [00:00<?, ?it/s]

Fold 5/5 done (8.0s).

SHAP stability -- raw Jaccard: 0.1178 +/- 0.0895
SHAP stability -- corr-aware: 0.1665 +/- 0.0705

Saved: xai_comparison_results_complete.json -- download and send this back.


In [ ]:
# --- IG top-20 gene ranking + beeswarm + COSMIC, for side-by-side comparison with SHAP ---

# 1. Full per-patient IG attributions (not just the mean) -- needed for the beeswarm plot
wrapped = WrappedTorchModel(model_full)
wrapped.eval()
X_t = torch.tensor(X_all.astype(np.float32), requires_grad=True)
baseline_vec = np.median(X_all, axis=0)
baseline = torch.tensor(np.tile(baseline_vec, (X_all.shape[0], 1)).astype(np.float32))
ig_full = IntegratedGradients(wrapped)
ig_attributions, _ = ig_full.attribute(X_t, baselines=baseline, n_steps=50, return_convergence_delta=True)
ig_attributions = ig_attributions.detach().numpy()

# 2. IG beeswarm (reuses shap.summary_plot -- it accepts any per-sample attribution array)
shap.summary_plot(ig_attributions, X_all, feature_names=feature_names, max_display=20, show=False)
plt.savefig('ig_beeswarm.png', dpi=500, bbox_inches='tight')
plt.close()
print("Saved: ig_beeswarm.png")

# 3. IG top-20 genes
ig_gene_scores = np.abs(ig_attributions[:, CLIN_DIM:]).mean(axis=0)
ig_top20_idx   = np.argsort(-ig_gene_scores)[:20]
ig_top20_genes = [(gene_cols[i], ig_gene_scores[i]) for i in ig_top20_idx]

print("\nTop 20 IG genes:")
for gene, val in ig_top20_genes:
    print(f"  {gene}: {val:.6f}")

# 4. COSMIC annotation for IG top-20 (reuses your existing cosmic_paad set)
print("\nTop 20 IG genes -- COSMIC PDAC driver annotation:")
print(f"{'Rank':<6}{'Gene':<15}{'Mean |IG|':<15}{'COSMIC Driver'}")
print("-" * 55)
for rank, (gene, val) in enumerate(ig_top20_genes, 1):
    is_driver = "YES" if gene in cosmic_paad else "-"
    print(f"{rank:<6}{gene:<15}{val:<15.6f}{is_driver}")

ig_overlap_cosmic = [g for g, _ in ig_top20_genes if g in cosmic_paad]
print(f"\nIG overlap with COSMIC PDAC drivers: {len(ig_overlap_cosmic)}/20 -> {ig_overlap_cosmic}")

# 5. Cross-method agreement: SHAP top-20 vs IG top-20
shap_gene_set = set(g for g, _ in top20_genes)   # from your existing SHAP cell
ig_gene_set   = set(g for g, _ in ig_top20_genes)
shared_genes  = shap_gene_set & ig_gene_set

print(f"\n=== SHAP vs IG top-20 gene overlap ===")
print(f"Shared genes ({len(shared_genes)}/20): {sorted(shared_genes)}")
print(f"SHAP-only: {sorted(shap_gene_set - ig_gene_set)}")
print(f"IG-only:   {sorted(ig_gene_set - shap_gene_set)}")